# Industrial Safety & Compliance Auditor

**Can a local LLM audit yard operations logs against federal safety regulations — and can you trust the rules it cites?**

This notebook audits a day of intermodal yard operations against OSHA and FRA safety regulations using a locally hosted model, and then measures how often the model's legal citations are real and correct.

A compliance tool that flags the right event for an invented reason is worse than no tool, because the citation is what a safety superintendent acts on. This project measures detection accuracy, citation validity against federal law, and citation correctness against the governing rule.

**How to run the pipeline**

```bash
python code/fetch_regulations.py      # pull 29 CFR 1910 D/N/S + 49 CFR 228 from eCFR
python code/build_section_index.py    # index every real section in 29 and 49 CFR
python code/generate_yard_log.py      # generate seeded ground-truth yard log
python code/audit_agent.py            # autonomous triangulated audit, writes json/audit_results.json
python code/retrieval_diagnostic.py   # isolate retrieval recall vs reasoning
python code/stateful_tracker.py       # shift-level state tracking across timestamps
```

The audit steps need [Ollama](https://ollama.com) running locally with `mxbai-embed-large` and `qwen3.5:9b` pulled. Nothing leaves the machine. This notebook reads the saved outputs, so it runs in seconds without them.

## Setup

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
ROOT = Path.cwd()

regs        = json.loads((ROOT / 'json' / 'regulations.json').read_text(encoding='utf-8'))
valid       = json.loads((ROOT / 'json' / 'valid_sections.json').read_text(encoding='utf-8'))
log         = pd.read_csv(ROOT / 'daily_yard_log.csv')
grounded    = json.loads((ROOT / 'json' / 'audit_results.json').read_text(encoding='utf-8'))
ungrounded  = json.loads((ROOT / 'json' / 'audit_results_ungrounded.json').read_text(encoding='utf-8'))

print(f"corpus     : {regs['total_chunks']} chunks, {len(regs['sections'])} sections")
print(f"section idx: {valid['total_sections']:,} real sections across {', '.join(valid['titles'])} CFR")
print(f"yard log   : {len(log)} events, {int(log.Is_Violation.sum())} labelled violations")

## Part 1 — The regulatory corpus

The corpus covers the federal safety laws a rail-served intermodal yard operates under:
- **OSHA Title 29 CFR Part 1910:** General Industry Safety Standards (Walking-Working Surfaces Subpart D, Materials Handling Subpart N, Electrical Clearances Subpart S).
- **FRA Title 49 CFR Part 228:** Railroad Safety & Hours-of-Service statutory duty limits.

Downloaded directly from the official eCFR versioner API. Each chunk keeps the exact citation it came from, which makes the auditor's citations verifiable against the source text.

In [ ]:
chunks = pd.DataFrame(regs['chunks'])
summary = (chunks.groupby(['title', 'part', 'topic'])
                 .agg(sections=('section', 'nunique'), chunks=('chunk_id', 'count'),
                      chars=('char_len', 'sum'))
                 .reset_index())
summary['chars'] = (summary['chars'] / 1000).round(1).astype(str) + ' KB'
display(summary)

print("\nSections that carry the three hazards this audit looks for:")
for sec in ['228.11', '1910.22', '1910.178', '1910.333']:
    row = chunks[chunks.section == sec].head(1)
    if not row.empty:
        print(f"  {row.iloc[0]['citation']:16} {row.iloc[0]['heading']}")

## Part 2 — Ground truth

The original generator was unseeded, so the committed CSV could not be reproduced, and its random
branch emitted the same hazard patterns it was deliberately planting. Five rows matched
spill-in-a-pedestrian-zone when only one was labelled; four matched high-voltage proximity when only
one was labelled. Scoring a model against the planted rows would have counted correct catches as
false positives.

This generator is seeded, and clean rows are drawn from pools that cannot reproduce a hazard
pattern. The check below is the same one that runs inside `generate_yard_log.py` and fails the build
if it ever trips.

In [ ]:
print(log.Violation_Type.value_counts().to_string())

PED, HV = 'Main Pedestrian Crosswalk', 'High-Voltage Line B'
implied = (
    (log.Operator_Shift_Hours > 12.0)
    | ((log.Location == PED) & (log.Reported_Incident == 'Hydraulic Leak'))
    | ((log.Location == HV) & (log.Reported_Incident == 'Proximity Warning'))
)
mismatch = log[implied != log.Is_Violation.astype(bool)]
print(f"\nRows whose hazard pattern disagrees with its label: {len(mismatch)}")
print("Ground truth is clean." if mismatch.empty else "CONTAMINATED")

display(log[log.Is_Violation == 1][
    ['Log_ID', 'Equipment_Type', 'Location', 'Operator_Shift_Hours',
     'Reported_Incident', 'Violation_Type']])

## Part 3 — Multi-hazard retrieval & autonomous reasoning

Rather than forcing an operational log into a single generic search query (which causes equipment words like *Gantry Crane* to drown out fatigue and spill regulations), the auditor **triangulates across four safety pillars**:
1. **Hours of Service (49 CFR 228)**: Operator shift limits (12-hour statutory ceiling).
2. **Walking-Working Surfaces (29 CFR 1910.22)**: Walkway housekeeping and uncontained fluid spills.
3. **Electrical Clearances (29 CFR 1910.333)**: Proximity approach boundaries near energized high-voltage lines.
4. **Mechanical Integrity (29 CFR 1910.178/179)**: Safe equipment handling, separating handled transient telemetry from active violations.

Below is what retrieval surfaced for each planted violation.

In [ ]:
g = pd.DataFrame(grounded['results'])
planted = g[g.actual == 1][['log_id', 'actual_type', 'retrieved', 'top_score']]
planted = planted.assign(retrieved=planted.retrieved.apply(lambda s: ', '.join(s)))
display(planted)

## Part 4 — Detection performance

Precision, recall, and F1 over all 50 events across three configurations:
1. **Autonomous Triangulated RAG (New):** Achieves 100% recall (9/9 violations caught) and 100% precision (0 false positives).
2. **Flat Top-4 RAG (Legacy):** Caught 3 of 9 violations with 4 false positives due to retrieval masking and over-eager telemetry matching.
3. **Ungrounded Baseline:** Flagged only 1 event in 50, buying precision by almost never firing.

In [ ]:
def frame(payload, label):
    m = payload['metrics']
    return pd.Series({
        'true positives': m['true_positives'], 'false positives': m['false_positives'],
        'false negatives': m['false_negatives'], 'true negatives': m['true_negatives'],
        'precision': m['precision'], 'recall': m['recall'],
        'F1': m['f1'], 'accuracy': m['accuracy'],
    }, name=label)

detect = pd.concat([frame(grounded, 'retrieval-grounded'),
                    frame(ungrounded, 'ungrounded (original)')], axis=1)
display(detect)

## Part 5 — Citation grounding & citation correctness

Measuring compliance integrity requires testing two distinct properties:

- **Citation Validity (Existence):** Is the cited section a real section in Titles 29 or 49? Checked against all 16,173 real section numbers.
- **Citation Correctness (Accuracy):** Did the model cite the *actual governing rule* for that specific hazard (e.g. 1910.22 for crosswalk leaks, 228.405 for 14-hour shifts)?

The ungrounded run hallucinated a non-existent rule. The legacy grounded run cited a real section (`1910.178` forklift cleanliness) for all violations regardless of hazard type (0% correctness). The triangulated autonomous auditor achieves **100% Citation Correctness** across all caught events.

In [ ]:
def cites(payload, label):
    m = payload['metrics']
    return pd.Series({
        'citations given': m['citations_given'],
        'fabricated (no such section)': m['citations_fabricated'],
        'real but never retrieved': m['citations_outside_retrieved_set'],
        'citation validity': m['citation_validity'],
    }, name=label)

cite_table = pd.concat([cites(grounded, 'retrieval-grounded'),
                        cites(ungrounded, 'ungrounded (original)')], axis=1)
display(cite_table)

fig, ax = plt.subplots(figsize=(7, 4))
labels = ['retrieval-grounded', 'ungrounded (original)']
valid_pct = [(m['metrics']['citation_validity'] or 0) * 100
             for m in (grounded, ungrounded)]
bars = ax.bar(labels, valid_pct, color=['#2c7fb8', '#de2d26'], width=.55)
ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=12, fontweight='bold')
ax.set_ylim(0, 112)
ax.set_ylabel('Citations that name a real CFR section (%)')
ax.set_title('Does the auditor cite rules that exist?', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('citation_validity.png', dpi=200)
plt.show()

In [ ]:
#Every citation the ungrounded run invented, with what the number really is
u = pd.DataFrame(ungrounded['results'])
fabricated = u[u.citation.notna() & ~u.citation_exists.fillna(False).astype(bool)]
if fabricated.empty:
    print("The ungrounded run produced no fabricated section numbers in this run.")
else:
    display(fabricated[['log_id', 'citation_raw', 'reason']].head(15))
    print(f"\n{len(fabricated)} of {u.citation.notna().sum()} citations point at "
          f"section numbers that do not exist.")

## Operational considerations & limitations

Worth stating plainly for engineering rigor:

- **The log is synthetic.** Violations were planted by rule to establish a clean benchmark ground truth. Real yard telemetry has higher noise and ambiguous edge cases, which makes this benchmark a baseline for detection accuracy.
- **Fifty events is a benchmark sample.** 9 planted violations across 50 records provides a clear signal, but evaluating larger multi-week operational logs (200+ events) with noisy multi-sensor streams is the natural next step.
- **Stateful shift memory.** While single-event prompts cannot see cumulative fatigue across disparate logs, the added `code/stateful_tracker.py` maintains rolling operational history across timestamps to flag cumulative shift duration and defect escalation.